In [ ]:
import tensorflow as tf
from tensorflow import data as tf_data

from tensorflow.keras import backend as K
import gc

import keras
import keras_cv
from keras_cv import bounding_box
from keras_cv import visualization

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms

import cv2
from pycocotools.coco import COCO

import os

In [ ]:
tf.test.is_gpu_available()
tf.test.gpu_device_name()

# Adjust the following to change YOLO model sizes

In [ ]:
mdl_path = "../../../saved_mdl/detect_yolo_xs/"
#mdl_path = "../../../saved_mdl/detect_yolo_s/"
#mdl_path = "../../../saved_mdl/detect_yolo_m/"
#mdl_path = "../../../saved_mdl/detect_yolo_l/"

yolo_mdl_str = "yolo_v8_xs_backbone_coco"
#yolo_mdl_str = "yolo_v8_s_backbone_coco"
#yolo_mdl_str = "yolo_v8_m_backbone_coco"
#yolo_mdl_str = "yolo_v8_l_backbone_coco"

In [ ]:
BATCH_SIZE = 16
LEARNING_RATE = 0.001
VAL_SIZE = 0.2
N_FOLDS = 5
GLOBAL_CLIPNORM = 10.0
EPOCHS = 500
# 200
SEED = 518

# Paths to the dataset and annotations
raw_img_path = "../../../data/lfa_single_ds/"
coco_file = "../../../data/lfa_single_ds/lfa_single_coco.json"


if not os.path.exists(mdl_path):
    os.makedirs(mdl_path)
filename_prefix = "yolo_kfold"

class_ids = ["Protein line", "RNA line", "Strip", "Control grid line", "Control line"]
class_mapping = dict(zip(range(len(class_ids)), class_ids))

obj_dl_resolution = [640, 640]

#### **Functions for handling coco format

In [ ]:
def parse_bbox_label(img_path, anns_id, coco):
    """Parses bounding boxes and labels from COCO annotations.
    Args:        img_path (str): Path to the image file.
        anns_id (list): List of annotation IDs for the image.
        coco (COCO): COCO object containing the annotations.
    Returns:
        tuple: A tuple containing:
            - bboxes: List of bounding boxes in the format [x_min, y_min, x_max, y_max].
            - labels: List of labels corresponding to the bounding boxes.
    """

    image = cv2.imread(img_path)
    original_height, original_width, _ = image.shape

    bboxes = []
    labels = []

    for ann in coco.loadAnns(anns_id):
        bbox = ann["bbox"]

        x_min = bbox[0]
        y_min = bbox[1]
        width = bbox[2]
        height = bbox[3]

        x_max = x_min + width
        y_max = y_min + height
        bboxes.append([x_min, y_min, x_max, y_max])

        labels.append(ann["category_id"] - 1)

        # print(bboxes)
    return bboxes, labels


def parse_coco_data(coco_annotation_file, jpg_filespath):
    """Parses the COCO annotations and returns image paths, labels, and bounding boxes.
    Args:
        coco_annotation_file (str): Path to the COCO annotation file.
        jpg_filespath (dict): Dictionary mapping image filenames to their full paths.
    Returns:
        list: A list containing three elements:
            - image_paths: List of image file paths.
            - labels: List of labels corresponding to the images.
            - bboxes: List of bounding boxes corresponding to the images.
    """
    coco = COCO(coco_annotation_file)
    image_ids = coco.getImgIds()

    image_paths = []
    bboxes = []
    labels = []

    for image_id in image_ids:
        img_info = coco.loadImgs([image_id])[0]

        # get the full path of the image file
        img_name = img_info["file_name"]
        img_path = jpg_filespath[img_name]

        # get the coco anno
        anns_id = coco.getAnnIds(image_id)

        bbox, label = parse_bbox_label(img_path, anns_id, coco)

        image_paths.append(img_path)
        bboxes.append(bbox)
        labels.append(label)

    return [image_paths, labels, bboxes]

### Get the path for all the jpg paths

In [ ]:
img_filespath = {}

print("Collecting all jpg/jpeg/png files...")
# Top directory
for dir_name in os.listdir(raw_img_path):
    dir_path = os.path.join(raw_img_path, dir_name)

    # Check if it is directory or other stuff
    if os.path.isdir(dir_path):
        # Iterate the folder inside
        for filename in os.listdir(dir_path):
            print(filename)
            if (
                filename.endswith(".jpg")
                or filename.endswith(".jpeg")
                or filename.endswith(".png")
            ):
                if filename in img_filespath:
                    print("Repeated filename!")
                else:
                    img_filespath[filename] = os.path.join(dir_path, filename)

                print(f"Found {filename} in {dir_path}")

In [ ]:
image_paths, labels, bboxes = parse_coco_data(coco_file, img_filespath)

In [ ]:
tfd_image_paths = tf.ragged.constant(image_paths)
tfd_labels = tf.ragged.constant(labels)
tfd_bboxes = tf.ragged.constant(bboxes)

num_samples = len(image_paths)


tf_ds = tf_data.Dataset.from_tensor_slices((tfd_image_paths, tfd_labels, tfd_bboxes))

# Shuffle the data during the training
tf_ds = tf_ds.shuffle(
    buffer_size=num_samples, seed=SEED, reshuffle_each_iteration=False
)
enum_tf_ds = tf_ds.enumerate()

### **Functions to support K-Fold with Tensorflow data

In [ ]:
def load_data_func(image_path, labels, bboxes):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    bounding_boxes = {
        "classes": tf.cast(labels, dtype=tf.float32),
        "boxes": bboxes,  # already float ragged
    }
    return {"images": tf.cast(image, tf.float32), "bounding_boxes": bounding_boxes}


def make_fold_datasets(enum_ds, k, fold_id, seed, batch_size, num_samples):
    # Select validation samples where (index % k) == fold_id
    print(f"Creating fold {fold_id} datasets...")
    val_raw_ds = enum_ds.filter(
        lambda i, data: tf.equal(tf.math.floormod(i, k), fold_id)
    ).map(lambda i, data: data, num_parallel_calls=tf.data.AUTOTUNE)

    # Training is everything else
    train_raw_ds = enum_ds.filter(
        lambda i, data: tf.not_equal(tf.math.floormod(i, k), fold_id)
    ).map(lambda i, data: data, num_parallel_calls=tf.data.AUTOTUNE)

    # Map -> batch -> prefetch
    print("Mapping datasets...")
    tf_train_ds = train_raw_ds.map(load_data_func, num_parallel_calls=tf.data.AUTOTUNE)
    tf_val_ds = val_raw_ds.map(load_data_func, num_parallel_calls=tf.data.AUTOTUNE)

    # Shuffle the training data before batching
    tf_train_ds = tf_train_ds.shuffle(
        buffer_size=num_samples, seed=seed, reshuffle_each_iteration=True
    )

    # Batch and prefetch the datasets
    print("Batching datasets...")
    tf_train_ds = tf_train_ds.ragged_batch(batch_size, drop_remainder=True).prefetch(
        tf.data.AUTOTUNE
    )
    tf_val_ds = tf_val_ds.ragged_batch(batch_size, drop_remainder=True).prefetch(
        tf.data.AUTOTUNE
    )

    return tf_train_ds, tf_val_ds

### ** Functions for visualization

In [ ]:
def visualize_dataset(inputs, value_range, rows, cols, bounding_box_format):
    inputs = next(iter(inputs.take(1)))
    images, bounding_boxes = inputs["images"], inputs["bounding_boxes"]

    # print(bounding_boxes)

    visualization.plot_bounding_box_gallery(
        images,
        value_range=value_range,
        rows=rows,
        cols=cols,
        y_true=bounding_boxes,
        scale=5,
        font_scale=0.7,
        bounding_box_format=bounding_box_format,
        class_mapping=class_mapping,
    )


def visualize_detections(model, dataset, bounding_box_format):
    for i in range(10):
        images, y_true = next(iter(dataset.take(i + 1)))
        print(type(images))
        y_pred = model.predict(images)
        y_pred = bounding_box.to_ragged(y_pred)
        visualization.plot_bounding_box_gallery(
            images,
            value_range=(0, 255),
            bounding_box_format=bounding_box_format,
            # y_true=y_true,
            y_pred=y_pred,
            scale=4,
            rows=2,
            cols=2,
            show=True,
            font_scale=0.7,
            class_mapping=class_mapping,
        )

## Augmentation and resizing, and finalizing train/test data

In [ ]:
augmenter = keras.Sequential(
    layers=[
        keras_cv.layers.RandomFlip(mode="horizontal", bounding_box_format="xyxy"),
        keras_cv.layers.Resizing(
            640, 640, bounding_box_format="xyxy", pad_to_aspect_ratio=True
        ),
        keras_cv.layers.JitteredResize(
            target_size=(640, 640), scale_factor=(0.8, 1.0), bounding_box_format="xyxy"
        ),
    ]
)

inference_resizer = keras_cv.layers.Resizing(
    640, 640, bounding_box_format="xyxy", pad_to_aspect_ratio=True
)


# Functions for the final dataset format: Desne instead of Tuple, GPU compatible
def dict_to_tuple(inputs):
    return inputs["images"], bounding_box.to_dense(
        inputs["bounding_boxes"],
    )


def make_train_val_data(
    tf_train_ds,
    tf_val_ds,
    visualize=False,
    augmenter=augmenter,
    inference_resizer=inference_resizer,
    dict_to_tuple=dict_to_tuple,
):
    """
    Build train/val pipelines:
      train:   train_ds -> augmenter -> dict_to_tuple -> prefetch
      val:     val_ds   -> inference_resizer -> dict_to_tuple -> prefetch
    """
    print("Creating train/val datasets...")
    autotune_tf = tf.data.AUTOTUNE
    autotune_ds = tf_data.AUTOTUNE

    print("Augmenting datasets...")
    tf_train_aug_ds = tf_train_ds.map(augmenter, num_parallel_calls=autotune_tf)
    tf_val_resize_ds = tf_val_ds.map(inference_resizer, num_parallel_calls=autotune_tf)

    if visualize:
        visualize_dataset(
            tf_train_aug_ds,
            bounding_box_format="xyxy",
            value_range=(0, 255),
            rows=2,
            cols=2,
        )

    train_data = tf_train_aug_ds.map(
        dict_to_tuple, num_parallel_calls=autotune_ds
    ).prefetch(autotune_ds)

    val_data = tf_val_resize_ds.map(
        dict_to_tuple, num_parallel_calls=autotune_ds
    ).prefetch(autotune_ds)

    return train_data, val_data

### YOLO

In [ ]:
class EvaluateCOCOMetricsCallback(keras.callbacks.Callback):
    def __init__(self, data, save_path):
        super().__init__()
        self.data = data
        self.metrics = keras_cv.metrics.BoxCOCOMetrics(
            bounding_box_format="xyxy",
            evaluate_freq=1e9,
        )

        self.save_path = save_path
        self.best_map = -1.0

    def on_epoch_end(self, epoch, logs):
        self.metrics.reset_state()
        for batch in self.data:
            images, y_true = batch[0], batch[1]
            y_pred = self.model.predict(images, verbose=0)
            self.metrics.update_state(y_true, y_pred)

        metrics = self.metrics.result(force=True)
        logs.update(metrics)

        current_map = metrics["MaP"]
        if current_map > self.best_map:
            self.best_map = current_map
            self.model.save(self.save_path)  # Save the model when mAP improves

        return logs


def get_tensorboard_callback(log_dir):
    """
    Create and return a TensorBoard callback.

    Args:
        log_dir (str): Directory to store TensorBoard logs.
        **kwargs: Any extra TensorBoard kwargs (e.g., histogram_freq=1, update_freq='batch').

    Returns:
        tf.keras.callbacks.TensorBoard
    """
    return tf.keras.callbacks.TensorBoard(log_dir=log_dir)


# Create functions for YOLOv8 model
def make_yolo_model(class_ids, fpn_depth=2):
    backbone = keras_cv.models.YOLOV8Backbone.from_preset(
        yolo_mdl_str, load_weights=True
    )

    # yolo_v8_l_backbone
    yolo = keras_cv.models.YOLOV8Detector(
        num_classes=len(class_ids),
        bounding_box_format="xyxy",
        backbone=backbone,
        fpn_depth=fpn_depth,  # as recommended by Keras
    )

    yolo.summary()

    return yolo


def make_test_yolo_model(mdl_path):
    backbone = keras_cv.models.YOLOV8Backbone.from_preset(
        yolo_mdl_str, load_weights=False
    )

    test_yolo_model = keras_cv.models.YOLOV8Detector(
        num_classes=len(class_ids),
        bounding_box_format="xyxy",
        backbone=backbone,
        fpn_depth=2,  # as recommended by Keras
    )
    test_yolo_model.load_weights(mdl_path)

    return test_yolo_model


def plt_train_history(history, plt_history_filename):
    """Plot the training history."""
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.plot(history.history["loss"], label="Training Loss")
    ax.plot(history.history["val_loss"], label="Validation Loss")

    # Find minimum validation loss and its epoch index
    min_loss = min(history.history["val_loss"])
    min_loss_epoch = history.history["val_loss"].index(min_loss)

    # Place the annotation at the top (y in axes coords), same x as the point
    trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.annotate(
        f"Min Loss: {min_loss:.4f}\nEpoch: {min_loss_epoch + 1}",
        xy=(min_loss_epoch, min_loss),
        xycoords="data",
        xytext=(min_loss_epoch, 0.98),
        textcoords=trans,
        arrowprops=dict(
            arrowstyle="->",
            lw=3.0,  # thicker arrow line
            mutation_scale=16,  # (optional) bigger arrowhead
            shrinkA=0,
            shrinkB=0,
        ),
        ha="center",
        va="bottom",
        fontsize=12,
        color="red",
    )

    ax.set_title("Training and Validation Loss")
    ax.set_xlabel("Epochs")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True)

    fig.savefig(plt_history_filename)
    plt.show()


def plt_mAP_history(history, plt_history_filename):
    """Plot the mAP history with a top-down arrow to the max mAP."""

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.plot(history.history["MaP"], label="Validation mAP")

    # Find maximum mAP and its epoch index
    map_vals = history.history["MaP"]
    max_map = max(map_vals)
    max_map_epoch = map_vals.index(max_map)

    # Place the annotation at the top (y in axes coords), same x as the point
    trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.annotate(
        f"Max mAP: {max_map:.4f}\nEpoch: {max_map_epoch + 1}",
        xy=(max_map_epoch, max_map),
        xycoords="data",
        xytext=(max_map_epoch, 0.98),  # near top of axes
        textcoords=trans,  # x in data, y in axes fraction
        arrowprops=dict(
            arrowstyle="->",
            lw=3.0,  # thicker arrow line
            mutation_scale=16,  # bigger arrowhead
            shrinkA=0,
            shrinkB=0,
        ),
        ha="center",
        va="bottom",
        fontsize=12,
        color="red",
    )

    ax.set_title("Validation mAP")
    ax.set_xlabel("Epochs")
    ax.set_ylabel("mAP")
    ax.legend()
    ax.grid(True)

    fig.savefig(plt_history_filename)
    plt.show()

In [ ]:
def reset_tf_keras():
    """Clear Keras/TensorFlow state and free memory."""
    try:
        K.clear_session()
    except Exception as e:
        print(f"[warn] K.clear_session() failed: {e}")
    try:
        gc.collect()
    except Exception as e:
        print(f"[warn] gc.collect() failed: {e}")
    try:
        plt.close("all")
    except Exception:
        pass

## Training using K-Fold

In [ ]:
all_histories = []
best_map_per_fold = []


# training loop over K folds N_FOLDS
for fold in range(0, N_FOLDS):

    print(rf"\n Placholder for Fold {fold}")
    tf_train_ds, tf_val_ds = make_fold_datasets(
        enum_tf_ds, N_FOLDS, fold, SEED, BATCH_SIZE, num_samples
    )

    train_data, val_data = make_train_val_data(tf_train_ds, tf_val_ds, True)

    yolo = make_yolo_model(class_ids)

    yolo.compile(
        classification_loss="binary_crossentropy",
        box_loss="ciou",
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,
            global_clipnorm=GLOBAL_CLIPNORM,
        ),
    )

    # Define the TensorBoard callback directory
    callbackdir = os.path.join(mdl_path, f"{filename_prefix}_{fold}_logs")
    tensorboard_callback = get_tensorboard_callback(
        callbackdir,
    )

    # Define the model save path
    h5_filename = os.path.join(mdl_path, f"{filename_prefix}_{fold}.h5")

    history = yolo.fit(
        x=train_data,  # train_data.take(20),
        validation_data=val_data,
        epochs=EPOCHS,
        callbacks=[
            EvaluateCOCOMetricsCallback(val_data, h5_filename),
            # tensorboard_callback
        ],
    )

    all_histories.append(history)

    fold_map = history.history.get("MaP", [])
    best_map_per_fold.append(max(fold_map) if len(fold_map) else float("nan"))

    # Plot the training history
    plt_train_history(
        history, os.path.join(mdl_path, f"{filename_prefix}_{fold}_loss.png")
    )
    # Plot the mAP history
    plt_mAP_history(
        history, os.path.join(mdl_path, f"{filename_prefix}_{fold}_map.png")
    )

    # plot some detections on val set
    test_yolo_model = make_test_yolo_model(h5_filename)
    visualize_detections(test_yolo_model, dataset=val_data, bounding_box_format="xyxy")

    # --- Reset TF/Keras & free memory after this fold ---
    try:
        yolo.reset_metrics()
    except Exception:
        pass
    try:
        del history
    except Exception:
        pass
    try:
        del train_data, val_data, tf_train_ds, tf_val_ds
    except Exception:
        pass
    try:
        del yolo
    except Exception:
        pass
    reset_tf_keras()

    # break